# ذخیره‌سازی داده‌های کووید‑۱۹ در SQLite به تفکیک قاره

این نوت‌بوک فایل `covid19_cleaned_final.csv` را خوانده و برای هر قاره (مانند Asia, Europe, ...) یک جدول مجزا در پایگاه‌داده‌ی `covid19_by_continent.db` ایجاد می‌کند.

In [4]:
import pandas as pd
import sqlite3
import re

In [5]:
# در صورت نیاز مسیر فایل را اصلاح کنید
df = pd.read_csv('covid19_cleaned_final.csv')
print(f'تعداد کل رکوردها: {len(df)}')
print(f'ستون‌های موجود: {list(df.columns)}')

تعداد کل رکوردها: 189
ستون‌های موجود: ['active', 'cases', 'continent', 'country', 'countryInfo', 'critical', 'deaths', 'population', 'recovered', 'tests', 'cfr', 'recovery_rate', 'active_rate', 'tests_per_capita']


In [8]:
# اتصال به دیتابیس (در صورت نبود، ایجاد می‌شود)
conn = sqlite3.connect('covid19_by_continent.db')

# دریافت قاره‌های یکتا
continents = df['continent'].unique()
print('قاره‌های موجود:', continents)

for cont in continents:
    # ساخت نام استاندارد جدول (حذف فاصله و کاراکترهای غیرمجاز)
    table_name = 'continent_' + re.sub(r'[^a-zA-Z0-9]', '_', cont).lower()
    
    # فیلتر داده‌های مربوط به آن قاره و حذف ستون 'continent' (چون در جدول جداگانه ذخیره می‌شود)
    df_cont = df[df['continent'] == cont].drop(columns=['continent'])
    
    # ذخیره در جدول (اگر جدول موجود باشد، بازنویسی می‌شود)
    df_cont.to_sql(table_name, conn, if_exists='replace', index=False)
    
    print(f'جدول "{table_name}" با {len(df_cont)} رکورد ساخته شد.')

# بستن اتصال
conn.close()
print('\nهمه‌ی جداول با موفقیت ذخیره شدند.')

قاره‌های موجود: <ArrowStringArray>
[             'Asia',            'Europe',            'Africa',
     'North America',     'South America', 'Australia-Oceania']
Length: 6, dtype: str
جدول "continent_asia" با 37 رکورد ساخته شد.
جدول "continent_europe" با 31 رکورد ساخته شد.
جدول "continent_africa" با 57 رکورد ساخته شد.
جدول "continent_north_america" با 36 رکورد ساخته شد.
جدول "continent_south_america" با 9 رکورد ساخته شد.
جدول "continent_australia_oceania" با 19 رکورد ساخته شد.

همه‌ی جداول با موفقیت ذخیره شدند.


In [7]:
conn = sqlite3.connect('covid19_by_continent.db')
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print('لیست جداول موجود در دیتابیس:')
for table in tables:
    print(f'  - {table[0]}')
conn.close()

لیست جداول موجود در دیتابیس:
  - continent_asia
  - continent_europe
  - continent_africa
  - continent_north_america
  - continent_south_america
  - continent_australia_oceania


In [10]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('covid19_by_continent.db')
df_asia = pd.read_sql_query("SELECT * FROM continent_asia", conn)
print(df_asia)
conn.close()

      active     cases                           country  \
0  -0.257922 -0.306262                       Afghanistan   
1  -0.283344  0.015378                           Armenia   
2  -0.308558  0.581946                        Azerbaijan   
3  -0.310975  0.425772                           Bahrain   
4   6.814741  2.376131                        Bangladesh   
5  -0.307263 -0.559660                            Bhutan   
6   0.041225 -0.144384                            Brunei   
7  -0.311176 -0.446752                          Cambodia   
8   0.108551  0.091438                             China   
9  -0.311186  0.354192                            Cyprus   
10  6.196122  2.098742                           Georgia   
11 -0.287620  3.688704                         Hong Kong   
12 -0.308812  2.991118                              Iraq   
13 -0.304596  1.929292                            Jordan   
14 -0.258399  1.434005                        Kazakhstan   
15  2.033407  0.333574                  